# 04 - SHAP Feature Importance Analysis

This notebook uses SHAP (SHapley Additive exPlanations) to interpret the EncryptionGuard
ML model. We compute SHAP values and generate beeswarm and bar plots for feature importance.

In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add backend to path for imports
sys.path.insert(0, str(Path("../backend")))

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Load the trained model
MODEL_PATH = Path("../backend/ml/artifacts/model.pkl")
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print(f"Model type: {type(model).__name__}")

# Load training data
from ml.train import load_data, create_splits

X, y = load_data()
X_train, X_test, y_train, y_test = create_splits(X, y)

# Get feature names
if hasattr(X_train, "columns"):
    feature_names = list(X_train.columns)
else:
    feature_names = [f"feature_{i}" for i in range(X_train.shape[1])]

print(f"Training samples: {X_train.shape[0]}")
print(f"Features: {len(feature_names)}")

## Compute SHAP Values

Using TreeExplainer for tree-based models (LightGBM, XGBoost, Random Forest, etc.).
TreeExplainer provides exact SHAP values efficiently for tree-based models.

In [ ]:
import shap

# Initialize TreeExplainer
explainer = shap.TreeExplainer(model)

# Compute SHAP values on test set
# Use a subset if the test set is large
max_samples = 1000
if len(X_test) > max_samples:
    X_sample = X_test.sample(n=max_samples, random_state=42)
    print(f"Using {max_samples} samples for SHAP computation.")
else:
    X_sample = X_test
    print(f"Using all {len(X_sample)} test samples.")

shap_values = explainer.shap_values(X_sample)

# Handle different SHAP value formats
if isinstance(shap_values, list):
    # Binary classification: use positive class SHAP values
    shap_vals = shap_values[1]
    print(f"SHAP values shape (positive class): {shap_vals.shape}")
else:
    shap_vals = shap_values
    print(f"SHAP values shape: {shap_vals.shape}")

## SHAP Summary Plot (Beeswarm)

The beeswarm plot shows the distribution of SHAP values for each feature.
Each dot represents one sample; color indicates the feature value (red=high, blue=low).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# SHAP beeswarm summary plot
shap.summary_plot(
    shap_vals,
    X_sample,
    feature_names=feature_names,
    show=False,
    max_display=20
)

plt.title("SHAP Summary Plot (Beeswarm)", fontsize=14)
plt.tight_layout()
plt.show()

## SHAP Bar Plot (Feature Importance)

The bar plot shows the mean absolute SHAP value for each feature,
providing a global measure of feature importance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# SHAP bar plot for feature importance
shap.summary_plot(
    shap_vals,
    X_sample,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20
)

plt.title("SHAP Feature Importance (Mean |SHAP value|)", fontsize=14)
plt.tight_layout()
plt.show()

# Print top features
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1][:10]
print("\nTop 10 features by mean |SHAP value|:")
for i, idx in enumerate(top_indices, 1):
    print(f"  {i}. {feature_names[idx]}: {mean_abs_shap[idx]:.4f}")